# 01. 데이터 전처리

머신러닝에 사용하기 전에 원본 데이터의 **결측치, 이상치, 자료형, 문자열 형식**을 확인하고 정리하는 과정을 학습합니다.

### 이번 노트에서 배우는 것
- `head()`, `info()`, `describe()`로 데이터 구조 확인
- `isnull().sum()`으로 결측치 확인
- 평균/중앙값을 이용한 결측치 대체
- IQR을 이용한 이상치 확인 및 제거
- 자료형과 문자열 형식 정리

> 데이터 확인 결과를 셀마다 반복 출력하지 않고, **한 단계가 끝날 때 한 번씩 검증**합니다.


In [ ]:
from common.libraries import *


## 1. 데이터 불러오기와 기본 구조 확인

처음 데이터를 받으면 바로 값을 수정하지 않고 먼저 전체 구조를 확인합니다.

- `head()` : 앞부분의 실제 값 확인
- `info()` : 행 수, 컬럼, 자료형, 결측치 여부 확인
- `describe()` : 평균·중앙값·최솟값·최댓값 등 기초 통계 확인
- `isnull().sum()` : 컬럼별 결측치 개수 확인


In [ ]:
df = pd.read_csv("./data/ecommerce_customers_noisy.csv")

display(df.head())
df.info()
display(df.describe(include="all"))
display(df.isnull().sum().rename("결측치 수"))


## 2. 결측치 처리

결측치는 모델 학습 전에 처리해야 합니다.

- `Age`는 극단값의 영향을 덜 받도록 **중앙값(median)** 으로 대체합니다.
- `MonthlyCharges`는 **평균값(mean)** 으로 대체합니다.
- `fillna()`의 결과를 원본 컬럼에 다시 저장해야 실제 데이터가 변경됩니다.


In [ ]:
age_median = df["Age"].median()
monthly_mean = df["MonthlyCharges"].mean()

df["Age"] = df["Age"].fillna(age_median)
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(monthly_mean)

print("Age 중앙값:", age_median)
print("MonthlyCharges 평균:", round(monthly_mean, 2))
display(df.isnull().sum().rename("처리 후 결측치 수"))


## 3. 이상치 처리 — IQR

IQR(Interquartile Range)은 1사분위수(Q1)와 3사분위수(Q3) 사이의 범위입니다.

일반적으로 다음 범위를 벗어난 값을 이상치 후보로 봅니다.

`Q1 - 1.5 × IQR ~ Q3 + 1.5 × IQR`

여기서는 `Age`에서 비정상적으로 큰 값을 확인하고 제거합니다.


In [ ]:
q1 = df["Age"].quantile(0.25)
q3 = df["Age"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (df["Age"] < lower_bound) | (df["Age"] > upper_bound)

print("IQR 범위:", lower_bound, "~", upper_bound)
print("이상치 개수:", outlier_mask.sum())

display(df.loc[outlier_mask, ["CustomerID", "Age"]].head())


In [ ]:
df = df.loc[~outlier_mask].copy()
df.reset_index(drop=True, inplace=True)

print("이상치 처리 후 데이터 크기:", df.shape)
display(df["Age"].describe())


## 4. 자료형과 문자열 형식 정리

결측치를 채운 `Age`는 다시 정수형으로 바꾸고, `Gender`는 대소문자가 섞이지 않도록 소문자로 통일합니다.

이처럼 **같은 의미의 값은 같은 형식으로 맞춰야** 이후 인코딩이나 집계에서 별개의 값으로 처리되지 않습니다.


In [ ]:
df["Age"] = df["Age"].astype(int)
df["Gender"] = df["Gender"].str.strip().str.lower()

print(df.dtypes)
display(df["Gender"].value_counts())


## 5. 전처리 결과 확인 및 저장

마지막에 결측치와 주요 범위를 한 번 더 확인한 뒤 다음 학습에서 사용할 정제 데이터를 저장합니다.


In [ ]:
print("남은 결측치:", int(df.isnull().sum().sum()))
print("Age 범위:", df["Age"].min(), "~", df["Age"].max())

display(df.head())

df.to_csv("./data/ecommerce_customers_clean.csv", index=False, encoding="utf-8-sig")


### 정리
- 원본 데이터는 **확인 → 처리 → 검증** 순서로 다룹니다.
- `fillna()`는 결과를 다시 저장하지 않으면 원본 데이터가 바뀌지 않습니다.
- 이상치는 무조건 삭제하는 것이 아니라 **도메인과 데이터 의미를 함께 보고 판단**해야 합니다.
- 전처리가 끝난 데이터는 다음 단계인 특성 엔지니어링에서 사용합니다.
